In [17]:
import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.12.4
Folder  : devge
  numpy          ok
  pandas         ok
  sklearn        ok

All good. You can carry on to Step 1.


In [19]:
import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("data/delivery.csv")


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> data\delivery.csv


In [21]:
import sys
from pathlib import Path

print("Python version :", sys.version.split()[0])
print("Python program :", sys.executable)

# A virtual environment is just a folder. If the path above sits
# inside a folder called .venv, you are in the course environment.
in_venv = ".venv" in sys.executable.replace("\\", "/")
print("Inside .venv    :", in_venv)
print("Working folder  :", Path.cwd())

Python version : 3.12.4
Python program : C:\Users\devge\.venv\Scripts\python.exe
Inside .venv    : True
Working folder  : C:\Users\devge


In [23]:
from importlib.metadata import version

LIBRARIES = ["numpy", "pandas", "scikit-learn", "matplotlib"]

for name in LIBRARIES:
    print(f"{name:<15} {version(name)}")

numpy           2.5.2
pandas          3.0.5
scikit-learn    1.9.0
matplotlib      3.11.1


In [25]:
WORK = Path("work")
WORK.mkdir(exist_ok=True)

lines = [f"{name}=={version(name)}" for name in LIBRARIES]
(WORK / "requirements.txt").write_text("\n".join(lines) + "\n",
                                       encoding="utf-8")

print("wrote", WORK / "requirements.txt")
print("-" * 40)
print((WORK / "requirements.txt").read_text(encoding="utf-8"))

wrote work\requirements.txt
----------------------------------------
numpy==2.5.2
pandas==3.0.5
scikit-learn==1.9.0
matplotlib==3.11.1



In [29]:
import numpy as np

careless = np.random.default_rng()   # no seed given
print("three random numbers:", np.round(careless.uniform(0, 10, 3), 2))

three random numbers: [6.69 4.38 2.31]


In [31]:
first  = np.random.default_rng(42).uniform(0, 10, 3)
second = np.random.default_rng(42).uniform(0, 10, 3)

print("first run :", np.round(first, 2))
print("second run:", np.round(second, 2))
print("identical :", np.array_equal(first, second))

first run : [7.74 4.39 8.59]
second run: [7.74 4.39 8.59]
identical : True


In [33]:
import hashlib

def sha256_of(path):
    """A short fingerprint of a file's exact contents."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

make_delivery_csv(WORK / "run_a.csv")
make_delivery_csv(WORK / "run_b.csv")

fp_a = sha256_of(WORK / "run_a.csv")
fp_b = sha256_of(WORK / "run_b.csv")

print("run A:", fp_a[:16], "...")
print("run B:", fp_b[:16], "...")
print("identical files:", fp_a == fp_b)

run A: 9e9f7a46c817d5bb ...
run B: 9e9f7a46c817d5bb ...
identical files: True


In [35]:
import pandas as pd

orders = pd.read_csv(DATA)

print("rows, columns:", orders.shape)
print()
print(orders.head())
print()
print(orders.describe().round(1))

rows, columns: (600, 5)

   distance_km  prep_time_min  traffic_level  rain  delivery_min
0         9.40             17              1     0          51.3
1         5.55             24              2     1          54.2
2        10.37             28              3     0          67.7
3         8.52             23              2     0          51.2
4         1.58             29              2     1          42.1

       distance_km  prep_time_min  traffic_level   rain  delivery_min
count        600.0          600.0          600.0  600.0         600.0
mean           6.2           17.6            2.0    0.3          46.6
std            3.3            7.4            0.8    0.4          12.1
min            0.6            5.0            1.0    0.0          18.5
25%            3.2           11.0            1.0    0.0          37.8
50%            6.2           17.0            2.0    0.0          46.5
75%            9.1           24.0            3.0    1.0          55.7
max           12.0      

In [37]:
import json

run_info = {
    "python": sys.version.split()[0],
    "seed": SEED,
    "rows": len(orders),
    "data_sha256": sha256_of(DATA),
    "libraries": {n: version(n) for n in LIBRARIES},
}

(WORK / "run_info.json").write_text(json.dumps(run_info, indent=2),
                                    encoding="utf-8")
print(json.dumps(run_info, indent=2))

{
  "python": "3.12.4",
  "seed": 42,
  "rows": 600,
  "data_sha256": "9e9f7a46c817d5bb81c3e458f66617d17a116a3218c6bb5f555c05dd831f59a1",
  "libraries": {
    "numpy": "2.5.2",
    "pandas": "3.0.5",
    "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1"
  }
}


In [41]:
import subprocess

def git(*args):
    """Run one git command inside work/ and show what it said."""
    done = subprocess.run(["git", *args], cwd=WORK,
                          capture_output=True, text=True)
    print("$ git", " ".join(args))
    print((done.stdout + done.stderr).strip() or "(no output)")
    print("-" * 50)
    return done

if not (WORK / ".git").exists():
    git("init", "-q")
git("config", "user.name", "SCSE3040 Student")
git("config", "user.email", "student@bennett.edu.in")
git("add", "requirements.txt", "run_info.json")
git("commit", "-q", "-m", "P01: pinned requirements and run record")
git("log", "--oneline")

$ git init -q
(no output)
--------------------------------------------------
$ git config user.name SCSE3040 Student
(no output)
--------------------------------------------------
$ git config user.email student@bennett.edu.in
(no output)
--------------------------------------------------
$ git add requirements.txt run_info.json
(no output)
--------------------------------------------------
$ git commit -q -m P01: pinned requirements and run record
(no output)
--------------------------------------------------
$ git log --oneline
1ff232a P01: pinned requirements and run record
--------------------------------------------------


CompletedProcess(args=['git', 'log', '--oneline'], returncode=0, stdout='1ff232a P01: pinned requirements and run record\n', stderr='')

In [45]:
T1_first_three = np.round(
    np.random.default_rng(7).uniform(0.5, 12.0, 600), 2
)[:3].tolist()

print("T1_first_three =", T1_first_three)

T1_first_three = [7.69, 10.82, 9.42]


In [47]:
MY_LIBS = ["numpy", "pandas", "scikit-learn"]
my_lines = "\n".join(f"{name}=={version(name)}" for name in MY_LIBS)
(WORK / "my_requirements.txt").write_text(my_lines + "\n")

print(my_lines)

numpy==2.5.2
pandas==3.0.5
scikit-learn==1.9.0


In [49]:
def fingerprint(path):
    return {
        "rows": len(pd.read_csv(path)),
        "sha256": sha256_of(path),
        "seed": SEED
    }

T3_fp = fingerprint(DATA)
print(T3_fp)

{'rows': 600, 'sha256': '9e9f7a46c817d5bb81c3e458f66617d17a116a3218c6bb5f555c05dd831f59a1', 'seed': 42}
